# Assignment 1 — Fraud Detection in Financial Transactions

**Dataset:** Bank Account Fraud (BAF) Dataset Suite, NeurIPS 2022 — `Base.csv` ([Kaggle](https://www.kaggle.com/datasets/sgpjesus/bank-account-fraud-dataset-neurips-2022))
**Task:** binary classification, target `fraud_bool` (1 = fraudulent bank-account application).
**Protocol:** 11 models (10 algorithm families from the assignment, XGBoost and LightGBM reported separately), Stratified 10-fold cross-validation, `random_state = 42`.

### Notebook structure
0. Settings (`QUICK_TEST`, `SAMPLE_SIZE`)
1. Environment & library versions
2. Data download (Kaggle API / kagglehub)
3. Exploratory data analysis (EDA)
4. Preprocessing (inside `sklearn.Pipeline` — no leakage between folds)
5. Models (incl. note on Ridge / Lasso / Elastic Net for classification)
6. Stratified 10-fold cross-validation
7. Table 1 + extended table
8. Visualisations (model comparison, PR/ROC curves, confusion matrices, feature importance, training time)
9. Summary of key numbers
10. Export `results.zip`

> Run in Google Colab with **Runtime → Run all**. The full run takes **≈ 1.5 h** on the free Colab CPU runtime (2 cores); `QUICK_TEST = True` takes ≈ 1 min.
> Outputs: `table1.csv / .xlsx / .md`, `figures/*.png` (300 dpi), `results/`, `results.zip`.

In [ ]:
# ===================== 0. SETTINGS =====================
# QUICK_TEST=True  -> fast smoke test (small sample, 3 folds) to check the notebook runs end-to-end.
# QUICK_TEST=False -> the full experiment reported in the article.
QUICK_TEST = False

SAMPLE_SIZE = 100_000      # stratified subsample of the 1M-row Base.csv (class ratio is preserved)
KNN_SAMPLE_SIZE = 30_000   # KNN gets its own smaller stratified subsample (prediction cost is O(n_train) per row)
N_SPLITS = 10              # k in Stratified k-fold
RANDOM_STATE = 42          # one seed for every random operation -> reproducible results
FORCE_RETRAIN = False      # False -> reuse cached CV results from results/ (useful if Colab disconnects)
SHOW_PLOTS = True          # False -> figures are only saved to figures/, not displayed

if QUICK_TEST:
    SAMPLE_SIZE, KNN_SAMPLE_SIZE, N_SPLITS = 5_000, 5_000, 3

print(f"QUICK_TEST={QUICK_TEST} | SAMPLE_SIZE={SAMPLE_SIZE:,} | KNN_SAMPLE_SIZE={KNN_SAMPLE_SIZE:,} | N_SPLITS={N_SPLITS}")

## 1. Environment

Colab ships `scikit-learn`, `xgboost`, `lightgbm`, `pandas`, `matplotlib`, `seaborn`, `openpyxl`; `catboost` and `kagglehub` are installed below only if missing. Library versions are printed so the results can be reproduced.

In [ ]:
# Install the packages Colab does not have by default (no-op if already installed)
import importlib.util, subprocess, sys
for pkg in ["catboost", "kagglehub", "openpyxl", "tabulate", "xgboost", "lightgbm"]:
    if importlib.util.find_spec(pkg) is None:
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

In [ ]:
import os, json, time, random, warnings, platform, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import sklearn
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score,
                             precision_recall_curve, roc_curve, confusion_matrix)
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (ExtraTreesClassifier, AdaBoostClassifier,
                              GradientBoostingClassifier, HistGradientBoostingClassifier)
import xgboost, lightgbm, catboost
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Fix every source of randomness
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

# Output folders
FIG_DIR, RES_DIR = Path("figures"), Path("results")
FIG_DIR.mkdir(exist_ok=True); RES_DIR.mkdir(exist_ok=True)

# Plot style: one neutral style for all figures, 300 dpi on save
sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
                     "axes.spines.top": False, "axes.spines.right": False, "grid.alpha": 0.35})
C_MAIN, C_ALT, C_THIRD = "#2a78d6", "#eb6834", "#1baf7a"   # blue = legit/main, orange = fraud, aqua = 3rd series
C_GRAY = "#8a8985"

def save_fig(fig, name):
    """Save a figure to figures/<name>.png at 300 dpi, then display (or just close) it."""
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path)
    if SHOW_PLOTS:
        plt.show()
    plt.close(fig)
    print(f"saved -> {path}")

versions = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "scikit-learn": sklearn.__version__, "xgboost": xgboost.__version__,
            "lightgbm": lightgbm.__version__, "catboost": catboost.__version__,
            "matplotlib": matplotlib.__version__, "seaborn": sns.__version__}
for k, v in versions.items():
    print(f"{k:<13} {v}")
print("CPU cores:", os.cpu_count())

## 2. Data download

Credentials are **never stored in the code**. The cell looks for them in this order:
1. **Colab Secrets** (key icon in the left sidebar): `KAGGLE_API_TOKEN` (new-style token), or `KAGGLE_USERNAME` + `KAGGLE_KEY` (legacy); enable *Notebook access*.
2. An existing `~/.kaggle/kaggle.json`.
3. If nothing is found, the notebook asks you to upload `kaggle.json` (Kaggle → Settings → API → *Create Legacy API Key*).

Only `Base.csv` (≈1M rows, 32 columns) is downloaded via `kagglehub`; the five biased variants (Variant I–V) are not used.

In [ ]:
KAGGLE_DATASET = "sgpjesus/bank-account-fraud-dataset-neurips-2022"
DATA_FILE = "Base.csv"

def setup_kaggle_credentials():
    """Put Kaggle credentials into environment variables without hard-coding them."""
    # 1) Colab Secrets
    try:
        from google.colab import userdata
        for name in ["KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"]:
            try:
                value = userdata.get(name)
                if value:
                    os.environ[name] = value.strip()
            except Exception:
                pass  # secret not defined / no notebook access
    except ImportError:
        pass  # not running in Colab
    if os.environ.get("KAGGLE_API_TOKEN") or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        print("Kaggle credentials: environment / Colab Secrets")
        return
    # 2) existing kaggle.json
    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    if kaggle_json.exists():
        print(f"Kaggle credentials: {kaggle_json}")
        return
    # 3) ask the user to upload kaggle.json
    try:
        from google.colab import files
        print("Upload your kaggle.json (Kaggle -> Settings -> API -> Create Legacy API Key)")
        uploaded = files.upload()
        if "kaggle.json" not in uploaded:
            raise RuntimeError("kaggle.json was not uploaded")
        kaggle_json.parent.mkdir(parents=True, exist_ok=True)
        kaggle_json.write_bytes(uploaded["kaggle.json"])
        kaggle_json.chmod(0o600)   # Kaggle requires the file to be private
        print(f"Kaggle credentials saved to {kaggle_json}")
    except ImportError:
        raise RuntimeError("No Kaggle credentials found. Set KAGGLE_API_TOKEN or create ~/.kaggle/kaggle.json")

def download_base_csv():
    """Download only Base.csv with kagglehub (cached between runs)."""
    import kagglehub
    try:
        path = kagglehub.dataset_download(KAGGLE_DATASET, path=DATA_FILE)   # single file
    except Exception as e:
        print("Single-file download failed, downloading the whole dataset:", e)
        path = Path(kagglehub.dataset_download(KAGGLE_DATASET)) / DATA_FILE
    return Path(path)

if Path(DATA_FILE).exists():          # a local copy next to the notebook has priority
    csv_path = Path(DATA_FILE)
else:
    setup_kaggle_credentials()
    csv_path = download_base_csv()
print("Data file:", csv_path, f"({csv_path.stat().st_size / 1e6:.0f} MB)")

df_full = pd.read_csv(csv_path)
print("Shape:", df_full.shape)

## 3. Exploratory data analysis

EDA is performed on the **full** 1M-row file (descriptive statistics only, no model fitting).

**Missing values in BAF.** The dataset has no `NaN`s — missing values are encoded with negative sentinels. According to the BAF datasheet (Jesus et al., 2022):

| Feature | Valid range | Missing encoding |
|---|---|---|
| `prev_address_months_count` | [-1, 380] | -1 |
| `current_address_months_count` | [-1, 406] | -1 |
| `intended_balcon_amount` | [-16, 114] | any negative value |
| `bank_months_count` | [-1, 32] | -1 |
| `session_length_in_minutes` | [-1, 107] | -1 |
| `device_distinct_emails_8w` | [-1, 2] | -1 |

Other columns with negative values (e.g. `velocity_6h`) are *not* documented as missing and are left unchanged.

In [ ]:
TARGET = "fraud_bool"
CAT_COLS = [c for c in df_full.columns if df_full[c].dtype == "object"]   # string-coded categorical features
# Missing values encoded as negative numbers (BAF datasheet)
MISSING_SENTINEL_COLS = ["prev_address_months_count", "current_address_months_count", "intended_balcon_amount",
                         "bank_months_count", "session_length_in_minutes", "device_distinct_emails_8w"]

overview = pd.DataFrame({
    "dtype": df_full.dtypes.astype(str),
    "n_unique": df_full.nunique(),
    "min": df_full.min(numeric_only=True),
    "max": df_full.max(numeric_only=True),
    "n_NaN": df_full.isna().sum(),
})
print(f"Rows: {len(df_full):,} | Columns: {df_full.shape[1]} | categorical: {len(CAT_COLS)} | NaN cells: {df_full.isna().sum().sum()}")
print("Categorical:", CAT_COLS)
overview

In [ ]:
# --- 3.1 Class balance ---
counts = df_full[TARGET].value_counts().sort_index()
FRAUD_RATE_FULL = counts[1] / counts.sum()
print(f"Legitimate: {counts[0]:,} | Fraud: {counts[1]:,} | fraud rate = {FRAUD_RATE_FULL:.4%} | imbalance 1:{counts[0] / counts[1]:.0f}")

fig, ax = plt.subplots(figsize=(5.5, 3.2))
bars = ax.bar(["Legitimate (0)", "Fraud (1)"], counts.values, color=[C_MAIN, C_ALT], width=0.55)
for b, v in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:,}\n({v / counts.sum():.2%})", ha="center", va="bottom", fontsize=9)
ax.set_yscale("log"); ax.set_ylabel("Number of applications (log scale)")
ax.set_title("Class distribution of fraud_bool (Base.csv, n = 1,000,000)")
ax.set_ylim(top=counts.max() * 8)
save_fig(fig, "fig01_class_distribution")

In [ ]:
# --- 3.2 Missing values (negative sentinels) and whether "missingness" itself is informative ---
rows = []
for col in MISSING_SENTINEL_COLS:
    is_missing = df_full[col] < 0
    rows.append({"feature": col,
                 "missing_%": 100 * is_missing.mean(),
                 "fraud_rate_if_missing_%": 100 * df_full.loc[is_missing, TARGET].mean(),
                 "fraud_rate_if_present_%": 100 * df_full.loc[~is_missing, TARGET].mean()})
missing_df = pd.DataFrame(rows).sort_values("missing_%", ascending=False).reset_index(drop=True)
missing_df.to_csv(RES_DIR / "eda_missing_values.csv", index=False)
display(missing_df.round(2))

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.barh(missing_df["feature"], missing_df["missing_%"], color=C_MAIN, height=0.6)
for y, v in enumerate(missing_df["missing_%"]):
    ax.text(v + 0.5, y, f"{v:.1f}%", va="center", fontsize=9)
ax.invert_yaxis(); ax.set_xlabel("Share of missing values, %")
ax.set_title("Missing values encoded as negative sentinels")
ax.set_xlim(0, missing_df["missing_%"].max() * 1.15)
save_fig(fig, "fig02_missing_values")

In [ ]:
# --- 3.3 Distributions of key numeric features by class (density, so both classes are comparable) ---
KEY_FEATURES = ["name_email_similarity", "credit_risk_score", "customer_age", "income",
                "proposed_credit_limit", "date_of_birth_distinct_emails_4w", "velocity_6h", "session_length_in_minutes"]
plot_df = df_full[KEY_FEATURES + [TARGET]].copy()
for col in MISSING_SENTINEL_COLS:                 # do not plot sentinel values as real numbers
    if col in plot_df:
        plot_df.loc[plot_df[col] < 0, col] = np.nan

fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, col in zip(axes.ravel(), KEY_FEATURES):
    data = plot_df[col].dropna()
    hi = data.quantile(0.99)                      # drop the top 1% (long right tail) for readability
    many_values = data.nunique() > 20
    bins = np.linspace(data.min(), hi, 40) if many_values else np.sort(data.unique())
    for cls, color, label in [(0, C_MAIN, "Legitimate"), (1, C_ALT, "Fraud")]:
        vals = plot_df.loc[plot_df[TARGET] == cls, col].dropna()
        if many_values:                           # continuous feature -> step histogram (density)
            ax.hist(vals[vals <= hi], bins=bins, density=True, histtype="step", lw=2, color=color, label=label)
        else:                                     # few distinct values -> share of each value
            share = vals.value_counts(normalize=True).reindex(bins, fill_value=0)
            offset = -0.2 if cls == 0 else 0.2
            ax.bar(np.arange(len(bins)) + offset, share.values, width=0.4, color=color, label=label)
            ax.set_xticks(np.arange(len(bins))); ax.set_xticklabels([f"{b:g}" for b in bins], rotation=45, fontsize=7)
    ax.set_title(col, fontsize=10); ax.set_yticks([])
axes[0, 0].legend(frameon=False)
fig.suptitle("Distributions of key features by class (normalised within each class; top 1% of values not shown)", y=1.01)
fig.tight_layout()
save_fig(fig, "fig03_feature_distributions")

In [ ]:
# --- 3.4 Fraud rate per category of each categorical feature ---
fig, axes = plt.subplots(1, len(CAT_COLS), figsize=(16, 3.6), sharey=True)
for ax, col in zip(axes, CAT_COLS):
    rate = df_full.groupby(col)[TARGET].mean().sort_values(ascending=False) * 100
    ax.bar(rate.index.astype(str), rate.values, color=C_MAIN, width=0.6)
    ax.axhline(FRAUD_RATE_FULL * 100, color=C_GRAY, ls="--", lw=1)
    ax.set_title(col, fontsize=10); ax.tick_params(axis="x", rotation=45, labelsize=8)
axes[0].set_ylabel("Fraud rate, %")
axes[-1].text(1.02, FRAUD_RATE_FULL * 100, "overall\nrate", transform=axes[-1].get_yaxis_transform(),
              color=C_GRAY, fontsize=8, va="center")
fig.suptitle("Fraud rate by category (dashed line = overall fraud rate)", y=1.03)
save_fig(fig, "fig04_categorical_fraud_rate")

In [ ]:
# --- 3.5 Correlations (Spearman: robust to skewed distributions; sentinels replaced by NaN) ---
num_df = df_full.drop(columns=CAT_COLS + ["device_fraud_count"]).copy()   # device_fraud_count is constant (0)
for col in MISSING_SENTINEL_COLS:
    num_df.loc[num_df[col] < 0, col] = np.nan
corr = num_df.sample(200_000, random_state=RANDOM_STATE).corr(method="spearman")   # 200k rows are enough for stable estimates

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, cmap="RdBu_r", vmin=-1, vmax=1, center=0, square=True,
            linewidths=0.3, cbar_kws={"shrink": 0.6, "label": "Spearman ρ"}, ax=ax)
ax.set_title("Spearman correlation matrix of numeric features")
save_fig(fig, "fig05_correlation_matrix")

target_corr = corr[TARGET].drop(TARGET).sort_values(key=np.abs, ascending=False)
fig, ax = plt.subplots(figsize=(7, 7))
ax.barh(target_corr.index, target_corr.values, color=np.where(target_corr.values > 0, C_ALT, C_MAIN), height=0.65)
ax.axvline(0, color="black", lw=0.8); ax.invert_yaxis()
ax.set_xlabel("Spearman ρ with fraud_bool (orange = positive, blue = negative)")
ax.set_title("Correlation of numeric features with the target")
save_fig(fig, "fig06_target_correlation")
print("Top-5 |ρ| with target:\n", target_corr.head(5).round(3).to_string())

## 4. Preprocessing

**Feature set.** 32 columns − target `fraud_bool` − `device_fraud_count` (constant 0 in `Base.csv`, carries no information) − `month` (the application month 0–7 is a *time index*: with random stratified folds it would only let models memorise month-specific fraud rates, which does not generalise to future months) → **29 input features** (24 numeric/binary + 5 categorical).

**Sentinels → NaN.** Negative sentinel values (Section 3) are converted to `NaN`. This step is deterministic and row-wise (it uses no statistics of the data), so performing it before the split cannot leak information.

**Everything that learns from data lives inside a `sklearn.Pipeline`** and is therefore re-fitted on the training part of each fold only:
- numeric: `SimpleImputer(strategy="median", add_indicator=True)` — the median is computed on the training fold; `add_indicator` adds a binary "was missing" column, because EDA shows that missingness itself is predictive (e.g. fraud rate 1.42% when `prev_address_months_count` is missing vs 0.31% when present);
- numeric, **only for linear models and KNN**: `StandardScaler` (L1/L2 penalties and Euclidean distances are scale-sensitive; tree ensembles are scale-invariant);
- categorical: `OneHotEncoder(handle_unknown="ignore")`.

**Why a subsample?** 1M rows × 10 folds × 11 models is far beyond a free Colab session (2 CPU cores). A **stratified** random subsample of `SAMPLE_SIZE = 100,000` rows keeps the fraud rate identical to the full file (≈1.1%, ≈1,100 fraud cases) — enough positives for stable 10-fold estimates (~110 fraud cases per validation fold). KNN is additionally evaluated on a smaller stratified subsample (`KNN_SAMPLE_SIZE = 30,000`, drawn from the same 100k sample), because its prediction time grows linearly with the training-set size; this is marked explicitly in Table 1.

In [ ]:
# --- 4.1 Feature matrix ---
DROP_COLS = [TARGET, "device_fraud_count", "month"]
df = df_full.copy()
for col in MISSING_SENTINEL_COLS:
    df.loc[df[col] < 0, col] = np.nan            # negative sentinel -> real missing value

X_all = df.drop(columns=DROP_COLS)
y_all = df[TARGET].to_numpy()
NUM_COLS = [c for c in X_all.columns if c not in CAT_COLS]
N_FEATURES = X_all.shape[1]
print(f"Input features: {N_FEATURES} ({len(NUM_COLS)} numeric/binary + {len(CAT_COLS)} categorical), targets: 1")

# --- 4.2 Stratified subsample (class ratio preserved) ---
if SAMPLE_SIZE < len(X_all):
    X, _, y, _ = train_test_split(X_all, y_all, train_size=SAMPLE_SIZE, stratify=y_all, random_state=RANDOM_STATE)
else:
    X, y = X_all, y_all
X = X.reset_index(drop=True)

# KNN: an even smaller stratified subsample drawn from the main sample (keeps the original row index of X)
if KNN_SAMPLE_SIZE < len(X):
    X_knn, _, y_knn, _ = train_test_split(X, y, train_size=KNN_SAMPLE_SIZE, stratify=y, random_state=RANDOM_STATE)
else:
    X_knn, y_knn = X, y

sample_info = pd.DataFrame({
    "rows": [len(y_all), len(y), len(y_knn)],
    "fraud_cases": [y_all.sum(), y.sum(), y_knn.sum()],
    "fraud_rate_%": [100 * y_all.mean(), 100 * y.mean(), 100 * y_knn.mean()],
}, index=["Base.csv (full)", "main sample", "KNN sample"])
display(sample_info.round(3))
del df

In [ ]:
# --- 4.3 Preprocessing pipeline (fitted inside every CV fold -> no leakage) ---
def make_preprocessor(scale: bool) -> ColumnTransformer:
    """Median imputation + missing indicators (+ standardisation) for numeric, one-hot for categorical."""
    num_steps = [("impute", SimpleImputer(strategy="median", add_indicator=True))]
    if scale:
        num_steps.append(("scale", StandardScaler()))
    return ColumnTransformer(
        [("num", Pipeline(num_steps), NUM_COLS),
         ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS)],
        verbose_feature_names_out=False)

# Quick sanity check of the encoded feature space
_enc = make_preprocessor(scale=True).fit(X.head(20_000))
print("Features after encoding:", len(_enc.get_feature_names_out()))

## 5. Models

### 5.1 Why Ridge, Lasso and Elastic Net appear as classifiers
The assignment lists Ridge, Lasso and Elastic Net, which in their textbook form are **regression** models (least squares + L2 / L1 / L1+L2 penalty). Our target is **binary**, so a regression output would have to be thresholded and would not give calibrated probabilities. We therefore use the standard classification counterparts that keep exactly the same *regularisation* idea:

| Assignment | Implementation | Loss | Penalty |
|---|---|---|---|
| Ridge | `RidgeClassifier` | least squares on {−1, +1} labels | L2 (‖w‖²) |
| Lasso | `LogisticRegression(penalty="l1", solver="saga")` | logistic (log-loss) | L1 (‖w‖₁) → sparse weights / feature selection |
| Elastic Net | `LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.5)` | logistic (log-loss) | 0.5·L1 + 0.5·L2 |

`saga` is the scikit-learn solver that supports both L1 and elastic-net penalties. `RidgeClassifier` has no `predict_proba`; its `decision_function` is used as the ranking score for ROC-AUC / PR-AUC (both metrics depend only on the ranking).

### 5.2 Class imbalance (≈1 fraud per 90 applications)
No resampling is used; instead each model re-weights the minority class inside its own training procedure, which is applied only to the training fold:

| Model | Mechanism |
|---|---|
| Ridge, Lasso, Elastic Net, Extra Trees, LightGBM, HistGradientBoosting | `class_weight="balanced"` (weight ∝ 1 / class frequency) |
| AdaBoost, Gradient Boosting (no `class_weight` argument) | `sample_weight = compute_sample_weight("balanced", y_train)` passed to `fit` |
| XGBoost | `scale_pos_weight = n_negative / n_positive`, computed on each training fold |
| CatBoost | `auto_class_weights="Balanced"` |
| KNN | no weighting mechanism in the algorithm; it is kept as a distance-based baseline |

### 5.3 Hyper-parameters
Reasonable, commonly used defaults are fixed in advance (no tuning) so that all algorithms are compared under the same protocol; tuning would require nested CV and is outside the scope of this assignment.

In [ ]:
# Each model spec: display name, estimator, whether it needs scaled inputs, and how class imbalance is handled.
#   weighting = "builtin"          -> class_weight / auto_class_weights already set in the estimator
#               "sample_weight"    -> balanced sample weights are passed to fit()
#               "scale_pos_weight" -> XGBoost ratio n_neg/n_pos computed on the training fold
#               "none"             -> KNN (no weighting possible)
RS = RANDOM_STATE
MODEL_SPECS = [
    dict(key="ridge", name="Ridge (RidgeClassifier)", scale=True, weighting="builtin",
         estimator=RidgeClassifier(alpha=1.0, class_weight="balanced", random_state=RS)),
    dict(key="lasso", name="Lasso (LogReg, L1)", scale=True, weighting="builtin",
         estimator=LogisticRegression(penalty="l1", solver="saga", C=1.0, class_weight="balanced",
                                      max_iter=2000, tol=1e-3, random_state=RS)),
    dict(key="enet", name="Elastic Net (LogReg, L1+L2)", scale=True, weighting="builtin",
         estimator=LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.5, C=1.0,
                                      class_weight="balanced", max_iter=2000, tol=1e-3, random_state=RS)),
    dict(key="knn", name="KNN", scale=True, weighting="none",
         estimator=KNeighborsClassifier(n_neighbors=25, weights="distance", n_jobs=-1)),
    dict(key="extra_trees", name="Extra Trees", scale=False, weighting="builtin",
         estimator=ExtraTreesClassifier(n_estimators=200, min_samples_leaf=5, max_features="sqrt",
                                        class_weight="balanced", n_jobs=-1, random_state=RS)),
    dict(key="adaboost", name="AdaBoost", scale=False, weighting="sample_weight",
         estimator=AdaBoostClassifier(n_estimators=200, learning_rate=0.5, random_state=RS)),
    dict(key="gb", name="Gradient Boosting", scale=False, weighting="sample_weight",
         estimator=GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=3,
                                              subsample=0.8, random_state=RS)),
    dict(key="xgb", name="XGBoost", scale=False, weighting="scale_pos_weight",
         estimator=XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=6, subsample=0.8,
                                 colsample_bytree=0.8, tree_method="hist", eval_metric="aucpr",
                                 n_jobs=-1, random_state=RS)),
    dict(key="lgbm", name="LightGBM", scale=False, weighting="builtin",
         estimator=LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, subsample=0.8,
                                  subsample_freq=1, colsample_bytree=0.8, class_weight="balanced",
                                  n_jobs=-1, verbose=-1, random_state=RS)),
    dict(key="catboost", name="CatBoost", scale=False, weighting="builtin",
         estimator=CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                                      auto_class_weights="Balanced", verbose=0, thread_count=-1,
                                      allow_writing_files=False, random_seed=RS)),
    dict(key="hgb", name="HistGradientBoosting", scale=False, weighting="builtin",
         estimator=HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, class_weight="balanced",
                                                  early_stopping=False, random_state=RS)),
]
SPEC = {s["key"]: s for s in MODEL_SPECS}
print(len(MODEL_SPECS), "models:", ", ".join(s["name"] for s in MODEL_SPECS))

## 6. Stratified 10-fold cross-validation

`StratifiedKFold(n_splits=10, shuffle=True, random_state=42)`: every fold keeps the ≈1.1% fraud rate, and the **same** folds are used for every model (except KNN, which uses its own subsample split with the same settings), so differences between algorithms are not caused by different splits.

For each fold the whole pipeline (imputer → [scaler] → encoder → model) is fitted on the 9 training folds and evaluated on the held-out fold. We record:
- **threshold-dependent metrics** at the default decision rule (`predict`, i.e. probability ≥ 0.5 / decision function ≥ 0): Accuracy, Precision, Recall, F1 — all for the positive class *fraud*;
- **threshold-free ranking metrics**: ROC-AUC and PR-AUC (average precision);
- **fit time** (training time of the pipeline on the training folds, seconds) and prediction time.

**Why Accuracy is misleading here.** With 1.1% fraud, a model that labels *every* application as legitimate reaches **98.9% accuracy** while detecting zero fraud. Accuracy is dominated by the majority class, so it is reported (the assignment requires it) but not used for model selection. Precision (how many alarms are real fraud), Recall (how much fraud is caught) and their harmonic mean F1 focus on the fraud class. PR-AUC summarises the precision–recall trade-off over *all* thresholds and, unlike ROC-AUC, is sensitive to the class prior — its random-classifier baseline equals the fraud rate (≈0.011), not 0.5. **PR-AUC is therefore the primary metric for ranking models.**

**Out-of-fold predictions.** The validation-fold scores of every fold are stored in one out-of-fold (OOF) vector per model. This is exactly what `cross_val_predict(pipeline, X, y, cv=skf, method="predict_proba")` returns, but obtained from the same fitted models, so no model is trained twice. OOF predictions are used for the PR / ROC curves and confusion matrices.

**Run time.** On the free Colab CPU runtime (2 cores) the full cross-validation takes ≈ 1.5 h; most of it is spent by Lasso and Elastic Net (the `saga` solver needs ≈ 3 min per fold on 90k rows) and Gradient Boosting (≈ 1.5 min per fold).

Results of every model are cached in `results/` — if the Colab session disconnects, re-running the notebook skips models that are already done (`FORCE_RETRAIN = True` recomputes everything).

In [ ]:
from sklearn.base import clone

def fit_pipeline(spec, X_tr, y_tr):
    """Build preprocessing + model pipeline, apply the class-imbalance strategy, fit it. Returns (pipe, seconds)."""
    pipe = Pipeline([("prep", make_preprocessor(spec["scale"])), ("model", clone(spec["estimator"]))])
    fit_params = {}
    if spec["weighting"] == "sample_weight":
        fit_params["model__sample_weight"] = compute_sample_weight("balanced", y_tr)
    elif spec["weighting"] == "scale_pos_weight":
        pipe.set_params(model__scale_pos_weight=(y_tr == 0).sum() / (y_tr == 1).sum())
    t0 = time.perf_counter()
    pipe.fit(X_tr, y_tr, **fit_params)
    return pipe, time.perf_counter() - t0

def ranking_score(pipe, X_):
    """Probability of fraud if available, otherwise the decision function (RidgeClassifier)."""
    if hasattr(pipe, "predict_proba"):
        return pipe.predict_proba(X_)[:, 1]
    return pipe.decision_function(X_)

def run_cv(spec, X_, y_):
    """Stratified k-fold CV for one model. Returns per-fold metrics and out-of-fold predictions (cached)."""
    cache = RES_DIR / f"cv_{spec['key']}_n{len(y_)}_k{N_SPLITS}.joblib"
    if cache.exists() and not FORCE_RETRAIN:
        print(f"{spec['name']}: loaded from cache {cache}")
        return joblib.load(cache)

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    oof_score = np.full(len(y_), np.nan)
    oof_pred = np.full(len(y_), -1)
    rows = []
    for fold, (tr, va) in enumerate(skf.split(X_, y_), start=1):
        pipe, fit_time = fit_pipeline(spec, X_.iloc[tr], y_[tr])
        t0 = time.perf_counter()
        score = ranking_score(pipe, X_.iloc[va])
        pred = pipe.predict(X_.iloc[va])
        pred_time = time.perf_counter() - t0
        oof_score[va], oof_pred[va] = score, pred
        rows.append({"fold": fold,
                     "accuracy": accuracy_score(y_[va], pred),
                     "precision": precision_score(y_[va], pred, zero_division=0),
                     "recall": recall_score(y_[va], pred, zero_division=0),
                     "f1": f1_score(y_[va], pred, zero_division=0),
                     "roc_auc": roc_auc_score(y_[va], score),
                     "pr_auc": average_precision_score(y_[va], score),
                     "fit_time_s": fit_time, "predict_time_s": pred_time})
        r = rows[-1]
        print(f"  fold {fold:>2}/{N_SPLITS}: F1={r['f1']:.4f}  PR-AUC={r['pr_auc']:.4f}  ROC-AUC={r['roc_auc']:.4f}  fit={fit_time:.1f}s")

    result = {"key": spec["key"], "name": spec["name"], "n_samples": len(y_), "y": y_,
              "folds": pd.DataFrame(rows), "oof_score": oof_score, "oof_pred": oof_pred}
    joblib.dump(result, cache)
    return result

In [ ]:
# Train and evaluate all 11 models (≈ 1.5 h on the free Colab CPU with SAMPLE_SIZE=100k; ≈ 1 min with QUICK_TEST)
results = {}
t_start = time.perf_counter()
for spec in MODEL_SPECS:
    X_, y_ = (X_knn, y_knn) if spec["key"] == "knn" else (X, y)
    print(f"\n=== {spec['name']}  (n = {len(y_):,}, {N_SPLITS}-fold) ===")
    results[spec["key"]] = run_cv(spec, X_, y_)
print(f"\nTotal wall time: {(time.perf_counter() - t_start) / 60:.1f} min")

# Per-fold metrics of all models in one long table
fold_metrics = pd.concat([r["folds"].assign(model=r["name"]) for r in results.values()], ignore_index=True)
fold_metrics.to_csv(RES_DIR / "cv_fold_metrics.csv", index=False)

## 7. Table 1 — results of Stratified 10-fold cross-validation

Values are **mean ± standard deviation over the folds**. Precision, Recall and F1 refer to the positive class (*fraud*) at the default decision threshold. RMSE and R² are regression metrics and are not applicable to a classification task ("—"). Exported to `table1.csv`, `table1.xlsx` (sheets *Table 1* and *Extended*) and `table1.md`.

In [ ]:
def mean_std(s, digits=4):
    return f"{s.mean():.{digits}f} ± {s.std():.{digits}f}"   # pandas std = sample std (ddof=1)

def algo_label(r):
    """Model name; KNN is marked with its smaller sample size."""
    return f"{r['name']} (n = {r['n_samples']:,})*" if r["key"] == "knn" else r["name"]

# ---------- Table 1 (exact assignment format, in the assignment's order of algorithms) ----------
table1 = pd.DataFrame([{
    "Algorithm": algo_label(r),
    "Number of features": N_FEATURES,
    "Number of targets": 1,
    "k-fold validation": f"Stratified {N_SPLITS}-fold",
    "Accuracy": mean_std(r["folds"]["accuracy"]),
    "Precision": mean_std(r["folds"]["precision"]),
    "Recall": mean_std(r["folds"]["recall"]),
    "F1": mean_std(r["folds"]["f1"]),
    "RMSE": "—",
    "R2": "—",
} for r in results.values()])

# ---------- Extended table: ranking metrics + training time, sorted by PR-AUC ----------
extended = pd.DataFrame([{
    "Algorithm": algo_label(r),
    "key": r["key"],
    "Samples": r["n_samples"],
    "ROC-AUC": mean_std(r["folds"]["roc_auc"]),
    "PR-AUC": mean_std(r["folds"]["pr_auc"]),
    "F1": mean_std(r["folds"]["f1"]),
    "Fit time per fold, s": round(r["folds"]["fit_time_s"].mean(), 2),
    "Total CV fit time, s": round(r["folds"]["fit_time_s"].sum(), 1),
    "_pr_auc": r["folds"]["pr_auc"].mean(),
} for r in results.values()]).sort_values("_pr_auc", ascending=False).reset_index(drop=True)
extended.insert(0, "Rank (PR-AUC)", np.arange(1, len(extended) + 1))
RANKED_KEYS = extended["key"].tolist()          # model keys from best to worst PR-AUC
extended_out = extended.drop(columns=["key", "_pr_auc"])

# ---------- Export ----------
NOTE = (f"* KNN is evaluated on a stratified subsample of {KNN_SAMPLE_SIZE:,} rows; all other models on "
        f"{SAMPLE_SIZE:,} rows (stratified subsample of Base.csv, fraud rate {100 * y.mean():.2f}%). "
        "Precision/Recall/F1 for the fraud class at the default threshold. RMSE and R2 are not applicable "
        "to classification.")
table1.to_csv("table1.csv", index=False, encoding="utf-8-sig")
extended_out.to_csv("table1_extended.csv", index=False, encoding="utf-8-sig")
with pd.ExcelWriter("table1.xlsx") as xw:
    table1.to_excel(xw, sheet_name="Table 1", index=False)
    extended_out.to_excel(xw, sheet_name="Extended", index=False)
with open("table1.md", "w", encoding="utf-8") as f:
    f.write("## Table 1. Stratified k-fold cross-validation results (mean ± std over folds)\n\n")
    f.write(table1.to_markdown(index=False) + "\n\n" + NOTE + "\n\n")
    f.write("## Table 2. Extended results: ranking metrics and training time (sorted by PR-AUC)\n\n")
    f.write(extended_out.to_markdown(index=False) + "\n")

pd.set_option("display.width", 250, "display.max_columns", 20)
print("TABLE 1\n" + table1.to_string(index=False))
print("\n" + NOTE)
print("\nEXTENDED TABLE\n" + extended_out.to_string(index=False))

## 8. Visualisations

All figures are saved to `figures/` at 300 dpi. Models are ordered by mean PR-AUC (best on top). "Top-3" = the three models with the highest mean PR-AUC.

In [ ]:
# --- 8.1 Model comparison: mean F1 and PR-AUC with ± std over folds ---
order = RANKED_KEYS[::-1]                                   # best model at the top of a horizontal bar chart
labels = [algo_label(results[k]) for k in order]
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharey=True)
for ax, metric, title in [(axes[0], "f1", "F1 (fraud class, threshold 0.5)"), (axes[1], "pr_auc", "PR-AUC (average precision)")]:
    means = [results[k]["folds"][metric].mean() for k in order]
    stds = [results[k]["folds"][metric].std() for k in order]
    ax.barh(labels, means, xerr=stds, color=C_MAIN, height=0.6,
            error_kw={"ecolor": "#333333", "elinewidth": 1, "capsize": 3})
    for yi, (m, s) in enumerate(zip(means, stds)):
        ax.text(m + s + 0.005, yi, f"{m:.3f}", va="center", fontsize=8)
    ax.set_title(title); ax.set_xlabel("mean ± std over folds")
    ax.set_xlim(0, max(np.array(means) + np.array(stds)) * 1.2)
axes[1].axvline(y.mean(), color=C_GRAY, ls="--", lw=1, label=f"random classifier = fraud rate ({y.mean():.3f})")
axes[1].legend(loc="lower right", fontsize=8, framealpha=0.95)
fig.suptitle(f"Model comparison, Stratified {N_SPLITS}-fold CV (sorted by PR-AUC)", y=1.01)
fig.tight_layout()
save_fig(fig, "fig07_model_comparison_f1_prauc")

In [ ]:
# --- 8.2 Fold-to-fold variability: boxplots of F1 and PR-AUC ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharey=True)
for ax, metric, title in [(axes[0], "f1", "F1 per fold"), (axes[1], "pr_auc", "PR-AUC per fold")]:
    data = [results[k]["folds"][metric].values for k in order]
    ax.boxplot(data, vert=False, widths=0.55, patch_artist=True,
               boxprops={"facecolor": "#cfe0f5", "edgecolor": C_MAIN},
               medianprops={"color": C_ALT, "lw": 2}, whiskerprops={"color": C_MAIN}, capprops={"color": C_MAIN},
               flierprops={"marker": "o", "markersize": 4, "markerfacecolor": C_MAIN, "markeredgecolor": "white"})
    for yi, d in enumerate(data, start=1):                   # individual folds as dots
        ax.scatter(d, np.full(len(d), yi) + np.random.uniform(-0.12, 0.12, len(d)), s=10, color="#333333", alpha=0.6, zorder=3)
    ax.set_title(title); ax.set_xlabel("score")
axes[0].set_yticks(range(1, len(order) + 1)); axes[0].set_yticklabels(labels)
fig.suptitle(f"Distribution of fold scores ({N_SPLITS} folds; orange line = median)", y=1.01)
fig.tight_layout()
save_fig(fig, "fig08_fold_boxplots")

In [ ]:
# --- 8.3 PR and ROC curves of the top-3 models (out-of-fold predictions) ---
TOP3 = RANKED_KEYS[:3]
TOP_COLORS = [C_MAIN, C_ALT, C_THIRD]

def oof(k):
    """y_true and OOF score of model k (KNN uses its own subsample)."""
    r = results[k]
    return r["y"], r["oof_score"]

fig, ax = plt.subplots(figsize=(6.5, 5.2))
for k, c in zip(TOP3, TOP_COLORS):
    yt, sc = oof(k)
    p, rcl, _ = precision_recall_curve(yt, sc)
    ax.plot(rcl, p, color=c, lw=2, label=f"{algo_label(results[k])} (AP = {average_precision_score(yt, sc):.3f})")
ax.axhline(y.mean(), color=C_GRAY, ls="--", lw=1, label=f"Random classifier (AP = {y.mean():.3f})")
ax.set_xlabel("Recall (share of fraud detected)"); ax.set_ylabel("Precision (share of alerts that are fraud)")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title("Precision–Recall curves, top-3 models (out-of-fold)")
ax.legend(frameon=False, fontsize=8, loc="upper right")
save_fig(fig, "fig09_pr_curves_top3")

fig, ax = plt.subplots(figsize=(6.5, 5.2))
for k, c in zip(TOP3, TOP_COLORS):
    yt, sc = oof(k)
    fpr, tpr, _ = roc_curve(yt, sc)
    ax.plot(fpr, tpr, color=c, lw=2, label=f"{algo_label(results[k])} (AUC = {roc_auc_score(yt, sc):.3f})")
ax.plot([0, 1], [0, 1], color=C_GRAY, ls="--", lw=1, label="Random classifier (AUC = 0.5)")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate (Recall)")
ax.set_title("ROC curves, top-3 models (out-of-fold)")
ax.legend(frameon=False, fontsize=8, loc="lower right")
save_fig(fig, "fig10_roc_curves_top3")
# Note: OOF AUC is computed on the pooled out-of-fold scores, so it can differ slightly from the mean of per-fold AUCs.

In [ ]:
# --- 8.4 Confusion matrices of the top-3 models (OOF predictions, default threshold) ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for ax, k in zip(axes, TOP3):
    r = results[k]
    cm = confusion_matrix(r["y"], r["oof_pred"], labels=[0, 1])
    cm_row = cm / cm.sum(axis=1, keepdims=True)              # row-normalised: share of each true class
    annot = np.array([[f"{cm[i, j]:,}\n({cm_row[i, j]:.1%})" for j in range(2)] for i in range(2)])
    sns.heatmap(cm_row, annot=annot, fmt="", cmap="Blues", vmin=0, vmax=1, cbar=False, ax=ax,
                xticklabels=["Legitimate", "Fraud"], yticklabels=["Legitimate", "Fraud"],
                linewidths=2, linecolor="white", annot_kws={"fontsize": 10})
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title(algo_label(r), fontsize=10)
fig.suptitle("Confusion matrices of the top-3 models (out-of-fold, counts and row %)", y=1.03)
fig.tight_layout()
save_fig(fig, "fig11_confusion_matrices_top3")

In [ ]:
# --- 8.5 Feature importance of the best model (permutation importance, model-agnostic) ---
# The best pipeline is re-fitted on the training part of the first CV fold; each raw input feature is then
# randomly shuffled in the held-out fold and the drop in PR-AUC is measured (n_repeats times).
# Works for any model (incl. HistGradientBoosting / KNN, which have no feature_importances_) and is measured
# on the 29 original features rather than on one-hot columns.
BEST = RANKED_KEYS[0]
spec = SPEC[BEST]
X_b, y_b = (X_knn, y_knn) if BEST == "knn" else (X, y)
tr, va = next(StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(X_b, y_b))
best_pipe, _ = fit_pipeline(spec, X_b.iloc[tr], y_b[tr])
pi = permutation_importance(best_pipe, X_b.iloc[va], y_b[va], scoring="average_precision",
                            n_repeats=2 if QUICK_TEST else 5, random_state=RANDOM_STATE, n_jobs=1)
importance = (pd.DataFrame({"feature": X_b.columns, "importance_mean": pi.importances_mean,
                            "importance_std": pi.importances_std})
              .sort_values("importance_mean", ascending=False).reset_index(drop=True))
importance.to_csv(RES_DIR / f"feature_importance_{BEST}.csv", index=False)

top = importance.head(15)[::-1]
fig, ax = plt.subplots(figsize=(7.5, 6))
ax.barh(top["feature"], top["importance_mean"], xerr=top["importance_std"], color=C_MAIN, height=0.6,
        error_kw={"ecolor": "#333333", "elinewidth": 1, "capsize": 3})
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Mean decrease in PR-AUC when the feature is shuffled")
ax.set_title(f"Permutation feature importance — {spec['name']} (top 15)")
save_fig(fig, "fig12_feature_importance_best")
print(importance.head(10).round(4).to_string(index=False))

In [ ]:
# --- 8.6 Training time (mean fit time per fold, log scale) ---
time_order = sorted(results, key=lambda k: results[k]["folds"]["fit_time_s"].mean())
t_mean = np.array([results[k]["folds"]["fit_time_s"].mean() for k in time_order])
t_std = np.array([results[k]["folds"]["fit_time_s"].std() for k in time_order])
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.barh([algo_label(results[k]) for k in time_order], t_mean, xerr=t_std, color=C_MAIN, height=0.6,
        error_kw={"ecolor": "#333333", "elinewidth": 1, "capsize": 3})
for yi, (v, s) in enumerate(zip(t_mean, t_std)):
    ax.text((v + s) * 1.15, yi, f"{v:.2f} s", va="center", fontsize=8)   # label to the right of the error bar
ax.set_xscale("log"); ax.set_xlabel("Mean fit time per fold, seconds (log scale)")
ax.set_title(f"Training time per fold (Colab CPU, {os.cpu_count()} cores)")
ax.set_xlim(right=(t_mean + t_std).max() * 5)
save_fig(fig, "fig13_training_time")

## 9. Summary of key numbers

Collected in `results/summary.json` (used to fill in the factual parts of the article).

In [ ]:
def metric_summary(k):
    f = results[k]["folds"]
    return {m: {"mean": round(f[m].mean(), 4), "std": round(f[m].std(), 4)}
            for m in ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "fit_time_s"]}

summary = {
    "quick_test": QUICK_TEST,
    "dataset": {"name": "Bank Account Fraud (BAF) NeurIPS 2022 - Base.csv", "kaggle": KAGGLE_DATASET,
                "rows": int(len(y_all)), "columns": int(df_full.shape[1]),
                "fraud_cases": int(y_all.sum()), "fraud_rate": round(float(y_all.mean()), 5)},
    "sample": {"rows": int(len(y)), "fraud_cases": int(y.sum()), "fraud_rate": round(float(y.mean()), 5),
               "knn_rows": int(len(y_knn))},
    "features": {"input": N_FEATURES, "numeric": len(NUM_COLS), "categorical": len(CAT_COLS),
                 "dropped": ["device_fraud_count", "month"], "missing_sentinel_cols": MISSING_SENTINEL_COLS},
    "cv": f"StratifiedKFold(n_splits={N_SPLITS}, shuffle=True, random_state={RANDOM_STATE})",
    "ranking_by_pr_auc": [results[k]["name"] for k in RANKED_KEYS],
    "best_model": {"name": results[RANKED_KEYS[0]]["name"], **metric_summary(RANKED_KEYS[0])},
    "top3": {results[k]["name"]: metric_summary(k) for k in TOP3},
    "best_by_f1": results[max(results, key=lambda k: results[k]["folds"]["f1"].mean())]["name"],
    "fastest_model": results[time_order[0]]["name"],
    "slowest_model": results[time_order[-1]]["name"],
    "top10_features_best_model": importance.head(10)["feature"].tolist(),
    "library_versions": versions,
}
with open(RES_DIR / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(json.dumps(summary, indent=2, ensure_ascii=False))

## 10. Export

All outputs (tables, figures, per-fold metrics, summary) are packed into `results.zip` and downloaded. The cached model results (`results/*.joblib`) are left out to keep the archive small.

In [ ]:
ZIP_PATH = Path("results.zip")
files_to_pack = ([Path(p) for p in ["table1.csv", "table1.xlsx", "table1.md", "table1_extended.csv"]]
                 + sorted(FIG_DIR.glob("*.png"))
                 + sorted(p for p in RES_DIR.glob("*") if p.suffix in {".csv", ".json"}))
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in files_to_pack:
        zf.write(p, arcname=str(p))
print(f"{ZIP_PATH} - {len(files_to_pack)} files, {ZIP_PATH.stat().st_size / 1e6:.1f} MB")
for p in files_to_pack:
    print("  ", p)

try:
    from google.colab import files
    files.download(str(ZIP_PATH))
except ImportError:
    print("Not in Colab - results.zip is in the working directory.")